# Calibracion GQ/H y MRDF

Usa las mismas funciones de costo heredadas de `old_dynaengine`, con optimizacion local multistart para que el ejemplo sea rapido. Para calibrar una curva dinamica aislada se fija explicitamente `sigma_vertical=100 kPa`; en columnas, ese esfuerzo se recalcula por segmento.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "dynaengine").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_DIR = ROOT / "examples" / "data"
MATERIALS_PATH = DATA_DIR / "section_01_materials.json"
materials = json.loads(MATERIALS_PATH.read_text(encoding="utf-8"))
print(f"Proyecto: {ROOT}")
print(f"Materiales cargados: {len(materials)}")

import pandas as pd
from dynaengine import DEFAULT_SIGMA_VERTICAL_KPA, CalibrationSettings, calibrate_dynamic_curve, evaluate_dynamic_curve

Proyecto: C:\Users\joel.alarcon\Desktop\_code\prismo\external\DynaEngine
Materiales cargados: 6


In [2]:
material = materials[0]
curve_spec = {
    **material["dynamic_model"],
    "sigma_vertical": DEFAULT_SIGMA_VERTICAL_KPA,
}
print(
    f"Curva directa para {material['material_name']} evaluada con "
    f"sigma_vertical={DEFAULT_SIGMA_VERTICAL_KPA:g} kPa"
)
curve = evaluate_dynamic_curve(curve_spec)
display(curve.as_frame().head())

Curva directa para Material de poza evaluada con sigma_vertical=100 kPa


,strain,ggmax,damping_percent,shear_stress_kpa
0,0.000001,0.995191,0.882620,0.009952
1,0.000001,0.994652,0.887601,0.011173
2,0.000001,0.994052,0.893194,0.012544
3,0.000001,0.993386,0.899471,0.014081
4,0.000002,0.992645,0.906518,0.015806


In [3]:
settings = CalibrationSettings(
    optimizer="scipy",
    scipy_starts=3,
    scipy_maxiter=30,
    random_state=1,
)

result = calibrate_dynamic_curve(
    curve.calibration_data(),
    gmax_pa=70_000_000,
    tau_max_pa=120_000,
    settings=settings,
)

display(pd.DataFrame([{
    "gamma_ref": result.gamma_ref,
    "theta": result.theta,
    "mrdf": result.mrdf,
    "dmin": result.dmin,
    "gqh_score": result.gqh_score,
    "mrdf_score": result.mrdf_score,
}]))

,gamma_ref,theta,mrdf,dmin,gqh_score,mrdf_score
0,0.001714,"{'theta_1': -1.9999098633356398, 'theta_2': -2...","{'P1': 0.8622542474570024, 'P2': 0.55003159460...",0.832783,-0.227196,-0.038177


In [4]:
display(pd.DataFrame({
    "strain": result.strain,
    "target_ggmax": result.target_ggmax,
    "calibrated_ggmax": result.calibrated_ggmax,
    "target_damping_percent": result.target_damping_percent,
    "calibrated_damping_percent": result.calibrated_damping_percent,
}).head())

,strain,target_ggmax,calibrated_ggmax,target_damping_percent,calibrated_damping_percent
0,0.000001,0.995191,0.998159,0.882620,0.867321
1,0.000001,0.994652,0.997921,0.887601,0.871901
2,0.000001,0.994052,0.997650,0.893194,0.877122
3,0.000001,0.993386,0.997342,0.899471,0.883080
4,0.000002,0.992645,0.996992,0.906518,0.889886
